# LLM-Guided Chip Placement on Ariane (Phases 2–6)

Training-free greedy (MaskPlace wiremask) + a lightweight LLM region advisor.
Mode B = the **133 hard SRAM macros**. Setup clones code+data from GitHub.

Pipeline: greedy baseline -> region mechanism check -> dry-run -> **full eval
(greedy vs LLM-text vs LLM-image)** with comparison table + curve.

gym 0.21 can't build on Py3.12, so we use real gym if present else a tiny shim.

## Cell 1 — Setup (clone repo, gym shim, protobuf)

In [ ]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone","--depth","1",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("pulled latest")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

try:
    import gym; print("real gym:",gym.__version__)
except Exception:
    gym=types.ModuleType("gym"); spaces=types.ModuleType("gym.spaces")
    class _E: pass
    class _D:
        def __init__(s,n): s.n=int(n)
        def contains(s,x):
            try: x=int(x)
            except: return False
            return 0<=x<s.n
    class _B:
        def __init__(s,low=0,high=1,shape=None,dtype=None): s.low,s.high,s.shape,s.dtype=low,high,shape,dtype
    gym.Env=_E; spaces.Discrete=_D; spaces.Box=_B; gym.spaces=spaces
    sys.modules["gym"]=gym; sys.modules["gym.spaces"]=spaces; print("gym shim")

try: subprocess.check_call([sys.executable,"-m","pip","install","-q","anthropic"])
except subprocess.CalledProcessError as e: print("anthropic skipped:",e)

for f in ["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py",
          "region_constraint.py","llm_guided_placement.py","visualize.py","evaluate.py",
          "ariane/netlist.pb.txt"]:
    assert os.path.exists(f),f"MISSING {f}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell 2 — Sanity check

In [ ]:
from place_db import PlaceDB
placedb=PlaceDB("ariane")
hard=sum(1 for n in placedb.node_info if placedb.node_info[n].get("is_hard"))
print("Nodes",len(placedb.node_info),"| Nets",len(placedb.net_info),
      "| Canvas",placedb.max_height,"| Hard",hard)
assert placedb.max_height==357

## Phase 2 — Greedy baseline (133 hard macros, deterministic order)

Baseline #1. `hard_order='area'` is reproducible run-to-run.

In [ ]:
import importlib, greedy_place; importlib.reload(greedy_place)
from greedy_place import greedy_place as run_place
res2=run_place(hard_only=True, hard_order="area", grid=224,
               save_fig="greedy_hard133.png", verbose=False)
print(f"greedy: HPWL={res2['hpwl']:.4e}  overlaps={res2['overlaps']}  placed={res2['placed']}")

## Phase 3 — Region mechanism check (no LLM)

Round-robin all 133 across the 9 regions: confirms regions are respected
(`out-of-region <= fallback`) and that a bad assignment worsens HPWL.

In [ ]:
import importlib, region_constraint; importlib.reload(region_constraint)
from place_db import PlaceDB
from comp_res import comp_res
from greedy_place import run_greedy, count_overlaps, select_hard_macros
from region_constraint import REGION_LABELS, check_region_compliance
pdb=PlaceDB("ariane"); pdb.node_id_to_name=select_hard_macros(pdb, order="area")
pnm=len(pdb.node_id_to_name)
regions={t:REGION_LABELS[t%9] for t in range(pnm)}
env,n,fb=run_greedy(pdb,pnm=pnm,grid=224,regions=regions,verbose=False)
hpwl,_=comp_res(pdb,env.node_pos,env.ratio)
nviol,_=check_region_compliance(env,regions,224)
print(f"round-robin: HPWL={hpwl:.4e} overlaps={count_overlaps(env)} fallback={fb} out-of-region={nviol}")
print("mechanism OK:", nviol<=fb)

## Phase 4 — Dry-run (FREE, no API)

Hill-climb plumbing on all 133 with random partial regions. Confirms iter-0
baseline, accept/reject, early-stop — without spending tokens.

In [ ]:
import importlib, llm_guided_placement; importlib.reload(llm_guided_placement)
from llm_guided_placement import run_llm_loop
_=run_llm_loop(advisor="dummy", grid=224, max_iters=8, patience=3, save_best_fig="dry_best.png")

## API key (needed for the paid cells below)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"]=UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("API key loaded")

## Phase 6 — Full evaluation: greedy vs LLM-text vs LLM-image

Runs the text-only and image loops (hill-climb), prints the comparison table,
and saves `eval_curve.png` + best layouts. The **text-vs-image** gap is the
headline ablation. (This is the main paid cell — ~10–30 API calls total.)

In [ ]:
import importlib, evaluate; importlib.reload(evaluate)
from evaluate import evaluate as run_eval
ev=run_eval(grid=224, hard_order="area", model="claude-sonnet-4-6",
            max_iters=15, patience=3, outdir="/kaggle/working", run_image=True, verbose=True)

## Results — table, curve, and best layouts

In [ ]:
from IPython.display import Image, display
print("HPWL vs iteration:"); display(Image("/kaggle/working/eval_curve.png"))
print("greedy baseline:");   display(Image("greedy_hard133.png"))
print("LLM text-only best:");display(Image("/kaggle/working/eval_text.png"))
print("LLM + image best:");  display(Image("/kaggle/working/eval_image.png"))